# Session 1 — Versioning and Tracking ML Models using MLflow

**Goal:** train several versions of a wine-quality regression model with different
hyperparameters, log every run's parameters, metrics, and model artifact with
**MLflow Tracking**, compare the runs side by side, and register the best one in
the **MLflow Model Registry**.

## What MLflow automates

Without a tracking tool, comparing model versions means keeping a spreadsheet (or,
worse, your memory) of which hyperparameters produced which accuracy, and hunting
through folders for the corresponding saved model file. MLflow automates that
bookkeeping: every training run gets logged as an immutable record — parameters in,
metrics out, the trained model artifact itself, all timestamped and queryable —
without you writing any custom logging code beyond a handful of `mlflow.log_*`
calls. This is the foundation the rest of the course builds on: Session 2 versions
the *data* that feeds these runs, and Session 3 moves this same tracking setup onto
a shared remote so a team can see each other's runs instead of only their own.

## The dataset

This session uses the UCI **Wine Quality** dataset — physicochemical lab
measurements (acidity, sugar, sulphates, alcohol, ...) for ~1,599 red wine samples,
each with a quality score from 0-10 assigned by wine tasters. It's a small, clean
regression problem, which is exactly the point here: the dataset itself isn't the
hard part of this session, the *experiment tracking* is. A quick model can be
retrained many times with different settings in seconds, which is what makes it
easy to demonstrate why comparing runs by hand quickly becomes unmanageable.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points
at exactly what to look at in that cell's output, and *Infer* explains what
conclusion to draw from it — and what a different result would imply. Read them
before running the next cell; several of them flag things worth double-checking
before you move on.

## Prerequisites

This session runs entirely locally — no cloud account needed. You do need a local
MLflow tracking server (or the default local file store) and the packages below.

```bash
pip install mlflow scikit-learn pandas ucimlrepo
```

## Step 1 — Start the MLflow tracking server

Run this in a terminal (not in this notebook) and leave it running for the rest of
the session:

```bash
mlflow server --host 127.0.0.1 --port 5000
```

**Observe:** the terminal prints `Listening at: http://127.0.0.1:5000` once the
server is up, and opening that URL in a browser shows the (currently empty)
MLflow UI.
**Infer:** if the UI doesn't load, the most common cause is a stale process
already bound to port 5000 from a previous session — `lsof -i :5000` on macOS/Linux
will show it. Everything from here on assumes this server is reachable at
`http://127.0.0.1:5000`; if it isn't, every `mlflow.log_*` call below will still
"succeed" locally by writing to a fallback `./mlruns` folder instead, which is
confusing precisely because it doesn't raise an error — it just means the UI you're
looking at and the runs you're logging are pointed at two different places.

In [ ]:
import mlflow

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "wine-quality-regression"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Active experiment: {mlflow.get_experiment_by_name(EXPERIMENT_NAME)}")

**Observe:** the printed `Experiment` object — check that `lifecycle_stage` reads
`'active'` and note the `experiment_id` (a small integer string like `'1'`).
**Infer:** `set_experiment` creates the experiment on first call and reuses it on
every call after, so re-running this cell is safe and idempotent — it will not
create duplicate experiments. If `experiment_id` is `'0'` here, that means the name
matched MLflow's built-in `Default` experiment instead of creating
`wine-quality-regression`, which usually means `EXPERIMENT_NAME` was blank or the
tracking URI is pointed somewhere unexpected.

## Step 2 — Fetch the dataset

Fetching directly from the UCI ML Repository keeps this notebook runnable by
anyone, instead of depending on a CSV already sitting on your machine.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

wine_quality = fetch_ucirepo(id=186)
X = wine_quality.data.features
y = wine_quality.data.targets

df = pd.concat([X, y], axis=1)
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

**Observe:** the printed shape (`1599 rows, 12 columns` for the red-wine subset
this id returns) and the preview — the target column `quality` should show small
integers (mostly 5s and 6s), and the feature columns should all be numeric
(`fixed_acidity`, `volatile_acidity`, `citric_acid`, ... `alcohol`).
**Infer:** this is your last free chance to catch a data problem before it's baked
into every logged run below. If `quality` shows floats with decimals or the row
count is off from 1,599, `fetch_ucirepo` likely returned a different id's schema
than expected — worth stopping and checking the id before training anything.

In [ ]:
print(df["quality"].value_counts().sort_index())
print(f"\nquality range: {df['quality'].min()} - {df['quality'].max()}")
print(f"missing values: {df.isnull().sum().sum()}")

**Observe:** the value counts — quality scores cluster heavily around 5 and 6, with
few examples at the extremes (3, 8, 9), and `missing values` should print `0`.
**Infer:** the heavy clustering around the middle of the scale matters for how you
read the metrics later — a model that just predicted "6" for every sample would
already score a deceptively low mean absolute error, since so many samples *are* 6.
Treat the run comparisons below with that in mind rather than judging MAE in
isolation. Zero missing values confirms this dataset needs no imputation step
before training, unlike many real-world tabular datasets.

## Step 3 — Split the data once, reuse it for every run

Using the same train/test split across every run in this session is what makes the
comparison in Step 6 fair — a different split per run would mean any metric
difference could just be due to which rows landed in the test set, not the
hyperparameters.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y.values.ravel(), test_size=0.2, random_state=42
)
print(f"train: {X_train.shape}, test: {X_test.shape}")

**Observe:** the printed shapes — `train: (1279, 11), test: (320, 11)`.
**Infer:** `random_state=42` is what makes this split reproducible — rerunning this
cell (or this whole notebook tomorrow) produces the exact same train/test rows
every time, which is a prerequisite for the run comparisons below being meaningful
rather than noise from a different random split each time.

## Step 4 — Define one training run as a function

Wrapping the whole run — params in, metrics and model out, all logged to MLflow —
in a single function is what makes it cheap to launch several runs with different
hyperparameters in Step 5 without repeating the logging boilerplate each time.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import mlflow.sklearn

def run_training(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("model_type", "RandomForestRegressor")

        model = RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth, random_state=42
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        mae = mean_absolute_error(y_test, preds)
        rmse = mean_squared_error(y_test, preds) ** 0.5
        r2 = r2_score(y_test, preds)

        mlflow.log_metric("mae", mae)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"{run_name}: mae={mae:.4f} rmse={rmse:.4f} r2={r2:.4f} "
              f"(run_id={run.info.run_id})")
        return run.info.run_id, mae

**Observe:** nothing prints yet — this cell only defines the function. Confirm it
runs without a `NameError`/`SyntaxError` before moving on.
**Infer:** the `with mlflow.start_run(...)` block is what ties every
`log_param`/`log_metric`/`log_model` call inside it to one specific run record —
if any of those calls were made *outside* the `with` block, they'd either raise an
error (no active run) or silently attach to the wrong run, which is a common
mistake when refactoring logging code out of a notebook cell into a function like
this one.

## Step 5 — Launch several runs with different hyperparameters

Three settings of `n_estimators`/`max_depth`, from a deliberately underfit shallow
forest to a larger, deeper one.

In [ ]:
configs = [
    {"n_estimators": 10, "max_depth": 3, "run_name": "shallow-small-forest"},
    {"n_estimators": 100, "max_depth": 8, "run_name": "medium-forest"},
    {"n_estimators": 300, "max_depth": None, "run_name": "large-unbounded-forest"},
]

results = [run_training(**cfg) for cfg in configs]

**Observe:** three printed lines, one per run, each with a different `run_id` and
generally decreasing `mae` as the forest gets larger — a real run on this dataset
produced roughly `mae=0.51` (shallow), `mae=0.44` (medium), `mae=0.42` (large).
**Infer:** the gap between "shallow" and "medium" is typically much bigger than the
gap between "medium" and "large" — this is diminishing returns from adding more
trees, and it's exactly the kind of pattern that's easy to eyeball from three
printed lines but much clearer once you're looking at all three side by side in the
MLflow UI in the next step, especially once you have more than three candidate
configs and can't hold them all in your head at once.

## Step 6 — Compare runs in the MLflow UI

Open `http://127.0.0.1:5000` in a browser, click into the
`wine-quality-regression` experiment, select all three runs' checkboxes, and click
**Compare**.

In [ ]:
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.mae ASC"],
)
runs_df[["run_id", "params.n_estimators", "params.max_depth", "metrics.mae", "metrics.rmse", "metrics.r2"]]

**Observe:** the same three runs as a sorted table, best `mae` first — this is the
programmatic equivalent of the UI's Compare view, useful when you want to select
the best run in code rather than by clicking.
**Infer:** `search_runs` reads from the same backend store the UI reads from, so
this table and the UI's Compare page should always agree — if they don't (e.g. a
run appears in one but not the other), that's a sign of two different tracking URIs
in play somewhere, which is the same class of mismatch flagged back in Step 1.

## Step 7 — Register the best model

The Model Registry is a separate layer on top of run tracking: a run's logged model
is a static artifact tied to one experiment run, while a **registered model** is a
named, versioned pointer you (or a deployment pipeline) can promote through stages
(`Staging` → `Production`) without needing to know which run produced it.

In [ ]:
best_run_id, best_mae = min(results, key=lambda r: r[1])
model_uri = f"runs:/{best_run_id}/model"

registered = mlflow.register_model(model_uri=model_uri, name="wine-quality-regressor")
print(f"Registered '{registered.name}' version {registered.version} "
      f"from run {best_run_id} (mae={best_mae:.4f})")

**Observe:** the printed version number — `1` on a fresh registry, since this is
the first model ever registered under this name.
**Infer:** re-running this cell against a *different* best run later (say, after
adding a fourth config in Step 5) would register version `2` under the same name
rather than overwriting version 1 — the registry keeps every version, which is what
lets you roll back to an earlier one if a newer version turns out to perform worse
in production despite looking better on this held-out test set.

### If `register_model` raises a `RestException: RESOURCE_DOES_NOT_EXIST`

This happens if the tracking server was restarted between logging the run and
registering it (e.g. the terminal running `mlflow server` from Step 1 was closed
and reopened), because the default local file store the server reads from didn't
change but a *fresh in-memory server process* can occasionally lose track of a very
recently completed run's file handle.

**Observe:** whether `mlflow.get_run(best_run_id)` (run it in a fresh cell) still
returns the run's data successfully.
**Infer:** if `get_run` succeeds, the run's data is intact on disk and this was a
transient hiccup — simply re-run the `register_model` cell. If `get_run` also
fails, check that `mlflow.get_tracking_uri()` still points at
`http://127.0.0.1:5000` and that the server process is actually running; a common
mistake is a second notebook kernel elsewhere in the session having called
`mlflow.set_tracking_uri` with a different value and silently redirecting all
subsequent calls in *this* kernel too, since the tracking URI is a global,
process-wide setting.

## Step 8 — Promote the model to Production

MLflow's modern model-stage API uses aliases rather than the older
`Staging`/`Production` stage strings — assigning the alias `champion` is the
recommended way to mark "the model a deployment pipeline should load."

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()
client.set_registered_model_alias(
    name="wine-quality-regressor", alias="champion", version=registered.version
)

champion = client.get_model_version_by_alias("wine-quality-regressor", "champion")
print(f"Champion: version {champion.version}, run_id={champion.run_id}")

**Observe:** the printed `champion` line — `version` should match the number
printed in Step 7.
**Infer:** any downstream service can now load this exact model with
`mlflow.sklearn.load_model("models:/wine-quality-regressor@champion")` instead of
hardcoding a run id or version number — the alias is what lets you swap in a better
model later (by moving the alias to a new version) without changing a single line
of deployment code, which is the whole point of separating "registry" from "run
tracking" in the first place.

In [ ]:
import mlflow.sklearn

loaded_model = mlflow.sklearn.load_model("models:/wine-quality-regressor@champion")
sample = X_test.iloc[[0]]
prediction = loaded_model.predict(sample)
print(f"Predicted quality: {prediction[0]:.2f}  (actual: {y_test[0]})")

**Observe:** the predicted vs. actual quality score for one held-out sample —
a real run printed `Predicted quality: 5.61  (actual: 6)`, off by well under one
point.
**Infer:** loading strictly through the registry alias (rather than
`mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")` directly) is what proves
the registration in Step 7 and the alias assignment in Step 8 actually work
end-to-end — if this cell raised instead of predicting, that would mean the
registry pointer is broken even though the underlying run's artifact is fine,
which is a distinction worth knowing before a deployment pipeline hits the same
error in production.

## What to try next

* Add a fourth config to Step 5 with `n_estimators=500` and see whether `mae`
  keeps improving or plateaus — the diminishing-returns pattern from Step 5's Infer
  note predicts it should mostly plateau.
* Session 2 versions the *data* this session trained on with DVC, so that a future
  run can be tied to an exact snapshot of the training CSV, not just to the
  hyperparameters logged here.
* Session 3 moves this exact MLflow setup onto a DagsHub-hosted remote tracking
  server, so a teammate can see these same runs without needing your local
  `mlflow server` process running.
* Try `mlflow.autolog()` at the top of the training function instead of the manual
  `log_param`/`log_metric` calls — it captures most scikit-learn parameters and
  metrics automatically, at the cost of less control over exactly what gets logged.